In [ ]:
import scanpy as sc
import numpy as np
from scmer import UmapL1  # 使用SCMER库提供的接口
import pandas as pd

# 读取数据
adataRaw = sc.read_h5ad("../output/pbmc_benchmark_data/pbmc_benchmark_s1d1.h5ad")
adataRaw.var_names_make_unique()

# 过滤细胞和基因
sc.pp.filter_cells(adataRaw, min_genes=3)
sc.pp.filter_genes(adataRaw, min_cells=200)

# 确保稀疏矩阵转为密集格式
adataRaw.X = adataRaw.X.toarray()

# 数据归一化
sc.pp.normalize_total(adataRaw, target_sum=1e4)
sc.pp.log1p(adataRaw)

# 检查数据的形状和格式是否正确
assert adataRaw.X.shape[0] > 0 and adataRaw.X.shape[1] > 0, "数据为空，请检查输入文件。"
assert isinstance(adataRaw.X, np.ndarray), "数据格式不正确，确保为密集数组。"

# 使用 SCMER 进行特征选择
# 调整参数以符合 SCMER 的实际需求
model = UmapL1.tune(
    target_n_features=500,  # 目标特征数量
    X=adataRaw.X,           # 输入表达矩阵
    perplexity=30,          # 可选：调整perplexity（默认为30）
    smallest_log10_fold_change=-3,  # 可选：调整log fold change（默认为-3）
    max_iter=10             # 可选：最大迭代次数（默认为10）
)

# 使用调优的模型对数据进行特征降维
selected_adata = model.transform(adataRaw)

# 将选定特征更新到adataRaw对象
adataRaw = selected_adata

# 聚类以识别稀有细胞群
from sklearn.cluster import DBSCAN

clustering = DBSCAN(eps=0.5, min_samples=10).fit(adataRaw.X)

# 添加聚类结果
adataRaw.obs["cluster"] = clustering.labels_

# 计算稀有细胞簇的占比
cluster_counts = adataRaw.obs["cluster"].value_counts(normalize=True)
rare_clusters = cluster_counts[cluster_counts < 0.05].index  # 小于5%的簇

# 标记稀有细胞
adataRaw.obs["rare_cells"] = "non-rare"
adataRaw.obs.loc[adataRaw.obs["cluster"].isin(rare_clusters), "rare_cells"] = "rare"

# 保存稀有细胞索引
adataRaw.obs["rare_cells"].to_csv("SCMER_rare_cells_indices.csv", index=False)

# 数据标准化和PCA降维
sc.pp.scale(adataRaw)
sc.tl.pca(adataRaw, svd_solver="arpack")

# UMAP可视化稀有细胞群
sc.pp.neighbors(adataRaw, use_rep="X_pca")
sc.tl.umap(adataRaw)
sc.pl.umap(adataRaw, color="rare_cells")


/Users/huangjinjin/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Calculating distance matrix and scaling factors...
Computing pairwise distances...
Using 6 threads...
Mean value of sigma: 0.756579
Done. Elapsed time: 10.09 seconds. Total: 10.09 seconds.
Iteration 0 with lasso = 1e-05 in [ 1e-08 , 0.01 ]... Creating model without batches...
Optimizing using OWLQN (because lasso is nonzero)...
0 loss (before this step): 2.7268331050872803 Nonzero (after): 7275 Elapsed time: 41.17 seconds. Total: 51.26 seconds.
1 loss (before this step): 2.177631139755249 Nonzero (after): 4220 Elapsed time: 36.72 seconds. Total: 87.99 seconds.
2 loss (before this step): 2.099177360534668 Nonzero (after): 3546 Elapsed time: 36.52 seconds. Total: 124.51 seconds.
3 loss (before this step): 2.0743050575256348 Nonzero (after): 3160 Elapsed time: 36.50 seconds. Total: 161.01 seconds.
